# Epidemiological models

This notebook is the summer4 equivalent of summer2's infection and mixing
pages (`examples/04-flow-types`, `examples/09-mixing-matrices`). It is written
for epidemiologists who already think in terms of susceptibles, infectious
people, contact rates, and who-mixes-with-whom.

We will:

1. Build a plain SIR epidemic and plot it.
2. Contrast **frequency-dependent** and **density-dependent** transmission.
3. Stratify by age and attach a **mixing matrix**.
4. Show why homogeneous mixing makes age-specific infectiousness
   unidentifiable — and how assortative mixing fixes that.
5. Show **generalised** FOI (`I / N ** exponent`) matching frequency at
   exponent 1 and density at exponent 0.
6. Weight infectiousness by compartment and age, then run a small TB
   reinfection model (four susceptible sources, each with its own
   relative susceptibility).

Every plot is accompanied by an assertion so the page stays a runnable test.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    GroupedOutput,
    Multiply,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"


def plot_compartments(res, title):
    """Plot S, I, R totals over time (summer2-style outputs plot)."""
    frame = res["comp"].to_pandas()
    # Columns are compartment labels; sum age strata when present.
    totals = {}
    for col in frame.columns:
        state_name = col.split("_")[0] if "_" in col else col
        totals.setdefault(state_name, 0.0)
        totals[state_name] = totals[state_name] + frame[col]
    return pd.DataFrame(totals, index=frame.index).plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


def plot_by_age(res, state_name, title):
    """Plot one disease state, one line per age band."""
    frame = res["comp"].select(state[state_name]).to_pandas()
    frame.columns = [c.split("_")[-1] for c in frame.columns]
    return frame.plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


## A plain SIR epidemic

Start with no age structure. People move $S \rightarrow I$ when they are
infected, and $I \rightarrow R$ when they recover. The infection rate depends
on how many people are currently infectious — that dependence is the
**force of infection** $\lambda$.

The infection flow is a `TransitionFlow` whose rate is a `ForceOfInfection`:
name the infectious compartments, attach a mixing matrix (here a trivial
whole-population one), and add recovery as an ordinary transition.


In [ ]:
state = Property("state", ("S", "I", "R"))
# ForceOfInfection always groups by a property; a single-trait "pop" stands in
# for the unstratified whole-population case.
pop = Property("pop", ("all",))
pmap1 = PropertyMap.from_property(state).stratify(pop)

infectious = state["I"]
mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
m = FlowModel(pmap1)
m.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=infectious,
            group_by=mixing.prop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=1.0,
        ),
    )
)
m.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))

y0 = np.zeros(pmap1.size)
y0[pmap1.select(state["S"])] = 990.0
y0[pmap1.select(state["I"])] = 10.0
plan = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 20.0, 201),
)
res = m.compile().run({}, y0, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")

i_peak = float(np.max(np.asarray(res["comp"].select(state["I"]).values.data)))
assert i_peak > 10.0, "epidemic should grow from the seed"
plot_compartments(res, "Unstratified SIR (frequency-dependent infection)")


## Frequency vs density transmission

Two common ways to write the force of infection (summer2's
`add_infection_frequency_flow` / `add_infection_density_flow`):

| Kind | Force of infection $\lambda$ | Typical use |
|---|---|---|
| **Frequency** | $\lambda = c \, I / N$ | Close-contact diseases; rate depends on the *proportion* infectious |
| **Density** | $\lambda = c \, I$ | Environmental / density-driven contact; rate grows with absolute numbers |

The number of new infections per day is then $\lambda \times S$ in both cases.

The practical difference shows up when population size changes: under frequency
dependence, doubling everyone leaves $\lambda$ the same; under density
dependence it doubles. Below we run the same seed epidemic both ways and plot
infectious prevalence side by side.


In [ ]:
def run_kind(kind, contact_rate):
    infectious = state["I"]
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    mm = FlowModel(pmap1)
    mm.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection(
                "infection",
                infectious=infectious,
                group_by=mixing.prop,
                mixing=mixing,
                kind=kind,
                contact_rate=contact_rate,
            ),
        )
    )
    mm.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
    return mm.compile().run({}, y0, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")


freq = run_kind(FOIKind.FREQUENCY, contact_rate=1.0)
# Density contact rate is per person, so a small number matches a similar peak.
dens = run_kind(FOIKind.DENSITY, contact_rate=1e-3)

i_freq = np.asarray(freq["comp"].select(state["I"]).values.data)[:, 0]
i_dens = np.asarray(dens["comp"].select(state["I"]).values.data)[:, 0]
ts = np.asarray(freq["comp"].times.values)
compare = pd.DataFrame({"frequency": i_freq, "density": i_dens}, index=ts)
assert float(np.max(i_freq)) > 10.0 and float(np.max(i_dens)) > 10.0
compare.plot(
    title="Infectious prevalence: frequency vs density",
    labels={"index": "time (days)", "value": "infectious people"},
)


## Age stratification and who mixes with whom

By default, models assume **homogeneous mixing**: every person contacts every
other person at the same rate. That is often wrong. Children may mix mostly
with other children; older adults may mix mostly among themselves
(**assortative** or with-like mixing).

A **mixing matrix** $K$ is an $N \times N$ table for $N$ strata. Following
summer2: **columns are infectors, rows are the infected**. So $K_{ab}$ is the
relative contact that stratum $a$ (row) receives from stratum $b$ (column).

For two age bands:

| | young (infector) | old (infector) |
|---|---|---|
| **young** (infected) | young → young | old → young |
| **old** (infected) | young → old | old → old |

Under frequency dependence the force of infection in band $a$ is

$$\lambda_a = c \sum_b K_{ab}\,\frac{I_b}{N_b}.$$


In [ ]:
age = Property("age", ("young", "old"))
pmap_age = PropertyMap.from_property(state).stratify(age)

# Mildly assortative: more within-age contact than between-age.
K_assort = np.array(
    [
        [0.8, 0.2],  # young infected by young, old
        [0.3, 0.7],  # old infected by young, old
    ]
)

infectious = state["I"]
mixing = MixingMatrix(age, K_assort, normalize="rows", check_reciprocal=False)
epi = FlowModel(pmap_age)
epi.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=infectious,
            group_by=mixing.prop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("beta"),
        ),
    )
)
epi.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))

y0_age = np.zeros(pmap_age.size)
y0_age[pmap_age.select(state["S"] & age["young"])] = 490.0
y0_age[pmap_age.select(state["S"] & age["old"])] = 490.0
# Seed infection in the young only — assortative mixing keeps more of the
# early wave in that band.
y0_age[pmap_age.select(state["I"] & age["young"])] = 20.0
y0_age[pmap_age.select(state["I"] & age["old"])] = 0.0

plan_age = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "foi": SaveRequest(GroupedOutput("infection")),
    },
    ts=np.linspace(0.0, 40.0, 81),
)
res_age = epi.compile().run(
    {"beta": 1.2}, y0_age, t0=0.0, t1=40.0, dt=0.1, save=plan_age, solver="euler"
)

plot_by_age(res_age, "I", "Infectious people by age (assortative mixing)")


### Inspecting the force of infection itself

summer4 saves $\lambda$ as a properly dimensioned trace — one value per age
band per time — so you can plot and query it without re-slicing a broadcast
array. Under assortative mixing and a young-only seed, $\lambda$ should be
higher in the young early on.


In [ ]:
foi_frame = res_age["foi"].to_pandas()
foi_frame.columns = list(age.traits)
assert res_age["foi"].dims == ("time", "age")
# Early in the epidemic, young λ exceeds old λ under assortative mixing.
early = foi_frame.iloc[5]
assert float(early["young"]) > float(early["old"])
foi_frame.plot(
    title="Force of infection λ by age (assortative mixing)",
    labels={"index": "time (days)", "value": "λ (per day)"},
)


## Why mixing structure matters for inference

A common modelling choice is a **homogeneous** mixing matrix — every entry
equal (after row normalisation, $K_{ab} = 1/N$). Then every age band sees
**exactly the same** force of infection at every time:

$$\lambda_a = c \sum_b \tfrac{1}{N}\,\frac{I_b}{N_b} = \lambda$$

for all $a$. Age-specific infectiousness weights cannot be told apart from a
rescaling of the overall contact rate: the data carry no age gradient in
$\lambda$. An **assortative** matrix restores band-specific $\lambda_a$ and
makes that gradient identifiable again.

The plots below make the contrast visible. We assert that the maximum
difference between age bands is exactly zero under homogeneous mixing, and
strictly positive under assortative mixing.


In [ ]:
def run_mixing(K, title_suffix):
    model = FlowModel(pmap_age)
    foi = ForceOfInfection(
        "infection",
        infectious=state["I"],
        group_by=age,
        kind=FOIKind.FREQUENCY,
        contact_rate=Param("beta"),
        mixing=MixingMatrix(age, K, normalize="none", check_reciprocal=False),
    )
    model.add_flow(TransitionFlow("infection", state["S"], state["I"], foi))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
    out = model.compile().run(
        {"beta": 0.8},
        y0_age,
        t0=0.0,
        t1=30.0,
        dt=0.1,
        save=plan_age,
        solver="euler",
    )
    vals = np.asarray(out["foi"].values.data)
    frame = out["foi"].to_pandas()
    frame.columns = list(age.traits)
    diff = float(np.max(np.abs(vals[:, 0] - vals[:, 1])))
    fig = frame.plot(
        title=f"λ by age — {title_suffix} (max |Δλ| = {diff:.4g})",
        labels={"index": "time (days)", "value": "λ (per day)"},
    )
    return out, diff, fig


K_hom = np.ones((2, 2)) / 2.0
K_ass = np.array([[0.9, 0.1], [0.1, 0.9]])

_, diff_hom, fig_hom = run_mixing(K_hom, "homogeneous mixing")
fig_hom


In [ ]:
_, diff_ass, fig_ass = run_mixing(K_ass, "assortative mixing")
assert diff_hom == 0.0, "homogeneous mixing must collapse λ across ages"
assert diff_ass > 0.0, "assortative mixing must produce age-specific λ"
print(f"homogeneous max |Δλ| = {diff_hom}")
print(f"assortative max |Δλ| = {diff_ass:.4f}")
fig_ass


## Age-varying infectiousness

Sometimes one age band is more infectious per infectious person (different
viral load, behaviour, or duration). summer2 put those weights on the
stratification; summer4 attaches them to the force of infection — a band does
not know what a pathogen is.

`normalize="population"` rescales the weights so their population-weighted
mean is 1. That breaks the redundancy between "everyone is twice as
infectious" and "double the contact rate", without needing a prior demography
run to supply population shares.


In [ ]:
infectious = state["I"]
mixing = MixingMatrix(age, K_ass, normalize="none", check_reciprocal=False)
infectiousness = {age["young"]: 0.7, age["old"]: 1.4}
epi_nu = FlowModel(pmap_age)
epi_nu.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=infectious,
            group_by=mixing.prop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("beta"),
            infectiousness=infectiousness,
            normalize_infectiousness="population",
        ),
    )
)
epi_nu.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))

res_nu = epi_nu.compile().run(
    {"beta": 0.8}, y0_age, t0=0.0, t1=40.0, dt=0.1, save=plan_age, solver="euler"
)

# Same weights ×2 — under population normalisation the trajectories match.
infectiousness2 = {age["young"]: 1.4, age["old"]: 2.8}
epi_nu2 = FlowModel(pmap_age)
epi_nu2.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=infectious,
            group_by=mixing.prop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("beta"),
            infectiousness=infectiousness2,
            normalize_infectiousness="population",
        ),
    )
)
epi_nu2.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
res_nu2 = epi_nu2.compile().run(
    {"beta": 0.8}, y0_age, t0=0.0, t1=40.0, dt=0.1, save=plan_age, solver="euler"
)
np.testing.assert_allclose(
    np.asarray(res_nu["comp"].values.data),
    np.asarray(res_nu2["comp"].values.data),
    atol=1e-5,
)
print("2× infectiousness weights ≡ same epidemic under normalize='population'")
plot_by_age(res_nu, "I", "Infectious by age (age-varying infectiousness)")


## Generalised force of infection

Frequency dependence is infectious over population (`I / N`); density is
infectious alone (`I`). Kiribati and tb_macro sit in between:
`I / N ** exponent` with a calibrated exponent. That is
`kind=FOIKind.GENERALISED`. Exponent 1 must match frequency bit-for-bit; exponent 0
must match density. The two bars in each pair should be identical.


In [ ]:
def foi_at(kind, *, exponent=None):
    foi = ForceOfInfection(
        "infection",
        infectious=state["I"],
        group_by=age,
        kind=kind,
        contact_rate=0.4,
        mixing=MixingMatrix(age, np.eye(2), check_reciprocal=False),
        **({} if exponent is None else {"exponent": exponent}),
    )
    model = FlowModel(pmap_age)
    model.add_flow(TransitionFlow("inf", state["S"], state["I"], foi))
    y = np.array([900.0, 800.0, 50.0, 40.0, 0.0, 0.0])
    return np.asarray(model.compile().observe(0.0, y, {}).captures["infection"].data)


freq = foi_at(FOIKind.FREQUENCY)
dens = foi_at(FOIKind.DENSITY)
gen1 = foi_at(FOIKind.GENERALISED, exponent=1.0)
gen0 = foi_at(FOIKind.GENERALISED, exponent=0.0)
np.testing.assert_array_equal(freq, gen1)
np.testing.assert_array_equal(dens, gen0)
print(f"frequency {freq}, generalised(1) {gen1}")
print(f"density   {dens}, generalised(0) {gen0}")

pd.DataFrame(
    {
        "frequency": freq,
        "generalised exp=1": gen1,
        "density": dens,
        "generalised exp=0": gen0,
    },
    index=list(age.traits),
).plot.bar(
    title="Generalised FOI: exp=1 ≡ frequency, exp=0 ≡ density",
    labels={"index": "age", "value": "force of infection"},
)


## Compartment and age infectiousness

A trait map such as `{age["young"]: 0.7}` weights a whole age band. Kiribati
needs a weight per compartment *and* age: subclinical disease is less
infectious, and nobody under 15 transmits. Pass a sequence of
`(selector, weight)` pairs. Each pair multiplies the compartments it matches,
before those compartments are summed into the age group. A trait map is sugar
for the same pairs.

The bars below are the force of infection in a frequency-dependent model with
identity mixing. Age `0` has both subclinical and clinical infectious people,
so the unweighted force there is positive. After the weights, age `0` contributes
nothing and age `15` is lower, because its subclinical cases count as `0.4`.

In [ ]:
disease = Property("disease", ("S", "sub", "clin"))
band = Property("band", ("0", "15"))
pmap_w = PropertyMap.from_property(disease).stratify(band)
y_w = np.zeros(pmap_w.size)
y_w[pmap_w.select(disease["S"] & band["0"])] = 100.0
y_w[pmap_w.select(disease["sub"] & band["0"])] = 10.0
y_w[pmap_w.select(disease["clin"] & band["0"])] = 8.0
y_w[pmap_w.select(disease["S"] & band["15"])] = 200.0
y_w[pmap_w.select(disease["sub"] & band["15"])] = 20.0
y_w[pmap_w.select(disease["clin"] & band["15"])] = 40.0


def force_by_age(infectiousness):
    """Frequency-dependent force of infection, one number per age band."""
    kwargs = {}
    if infectiousness is not None:
        kwargs["infectiousness"] = infectiousness
    foi = ForceOfInfection(
        "infection",
        infectious=disease["sub"] | disease["clin"],
        group_by=band,
        kind=FOIKind.FREQUENCY,
        contact_rate=0.3,
        mixing=MixingMatrix(band, np.eye(2), normalize="none", check_reciprocal=False),
        **kwargs,
    )
    model = FlowModel(pmap_w)
    model.add_flow(TransitionFlow("inf", disease["S"], disease["sub"], foi))
    return np.asarray(model.compile().observe(0.0, y_w, {}).captures["infection"].data)


plain_w = force_by_age(None)
weighted_w = force_by_age([(disease["sub"], 0.4), (band["0"], 0.0)])
# The trait-map sugar for "age 0 is silent" matches the one-pair sequence.
age0_only = force_by_age({band["0"]: 0.0})
age0_pairs = force_by_age([(band["0"], 0.0)])
np.testing.assert_array_equal(age0_only, age0_pairs)
np.testing.assert_allclose(weighted_w[0], 0.0, atol=1e-12)
assert plain_w[0] > 0.0
assert weighted_w[1] < plain_w[1]
print(f"unweighted {plain_w}")
print(f"weighted   {weighted_w}")

pd.DataFrame(
    {
        "everyone fully infectious": plain_w,
        "subclinical 0.4×, age 0 silent": weighted_w,
    },
    index=list(band.traits),
).plot.bar(
    title="Force of infection by age: compartment weights before the group sum",
    labels={"index": "age", "value": "force of infection"},
)

## Reinfection from four susceptible states

tb_macro's force of infection is per age, `I / N ** exponent`, and each
susceptible source is infected at its own relative susceptibility. The model
below has four sources (`naive`, `latent`, `recovered`, `treated`) flowing into
`incipient`, one generalised force of infection, and `Multiply(rel_sus)` on
each infection flow. Clinical disease under age 0 does not transmit.

The figure is the four adult source compartments. A higher relative
susceptibility should finish lower: `naive` (1) below `latent` (0.5) below
`recovered` (0.2) below `treated` (0.1).

In [ ]:
src = Property(
    "src",
    ("naive", "latent", "recovered", "treated", "incipient", "clinical"),
)
age_tb = Property("age_tb", ("0", "15"))
pmap_tb = PropertyMap.from_property(src).stratify(age_tb)
rel_sus = {"naive": 1.0, "latent": 0.5, "recovered": 0.2, "treated": 0.1}
foi_tb = ForceOfInfection(
    "infection",
    infectious=src["clinical"],
    group_by=age_tb,
    kind=FOIKind.GENERALISED,
    exponent=0.5,
    contact_rate=0.008,
    infectiousness=[(age_tb["0"], 0.0)],
    mixing=MixingMatrix(age_tb, np.eye(2), normalize="none", check_reciprocal=False),
)
tb = FlowModel(pmap_tb)
for name, weight in rel_sus.items():
    tb.add_flow(
        TransitionFlow(
            f"infect_{name}",
            src[name],
            src["incipient"],
            foi_tb,
            adjust=(Multiply(weight),),
        )
    )
tb.add_flow(TransitionFlow("progress", src["incipient"], src["clinical"], 0.05))

y0_tb = np.zeros(pmap_tb.size)
for name in rel_sus:
    y0_tb[pmap_tb.select(src[name] & age_tb["0"])] = 40.0
    y0_tb[pmap_tb.select(src[name] & age_tb["15"])] = 200.0
y0_tb[pmap_tb.select(src["clinical"] & age_tb["0"])] = 25.0
y0_tb[pmap_tb.select(src["clinical"] & age_tb["15"])] = 15.0

plan_tb = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 60.0, 61),
)
res_tb = tb.compile().run({}, y0_tb, t0=0.0, t1=60.0, dt=0.1, save=plan_tb, solver="euler")
adult_sources = {}
for name in rel_sus:
    series = res_tb["comp"].select(src[name] & age_tb["15"]).to_pandas()
    adult_sources[name] = series.iloc[:, 0]
adult_frame = pd.DataFrame(adult_sources)
final_sources = adult_frame.iloc[-1]
assert final_sources["naive"] < final_sources["latent"]
assert final_sources["latent"] < final_sources["recovered"]
assert final_sources["recovered"] < final_sources["treated"]
# Age 0 is seeded with clinical disease but does not transmit, so its naive
# compartment is untouched.
naive_child = res_tb["comp"].select(src["naive"] & age_tb["0"]).to_pandas().iloc[:, 0]
np.testing.assert_allclose(naive_child.iloc[-1], naive_child.iloc[0], atol=1e-6)
print(final_sources.to_string())
adult_frame.plot(
    title="Adult sources reinfected at different relative susceptibility",
    labels={"index": "time (days)", "value": "people"},
)

## Summary

| Idea | summer2 | summer4 |
|---|---|---|
| Frequency / density infection | `add_infection_*_flow` | `ForceOfInfection(kind=...)` |
| Mixing matrix | `Stratification.set_mixing_matrix` | `MixingMatrix` |
| Infectiousness by compartment and age | `add_infectiousness_adjustments` | `infectiousness=[(selector, weight), ...]` (a trait map is sugar for the same pairs) |
| Generalised force of infection | `add_infection_generalised_flow` | `ForceOfInfection(kind=FOIKind.GENERALISED, exponent=...)` |
| Relative susceptibility of a source | per-flow multiplier | `TransitionFlow(..., adjust=Multiply(rel_sus))` |
| Inspect $\lambda$ | derived output | `GroupedOutput("infection")` → `dims=("time", "age")` |

For a full age-stratified SEIRS with calibration, see the case study under
`docs/case-studies/age-stratified-seirs`.
